# device check

In [1]:
from pathlib import Path
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights,resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA L4


# path

In [2]:
DATA_ROOT = Path(r"/home/sergiocloudwork/dataset/sampled_500")
TRAIN_DIR = DATA_ROOT / "train_mini"
VAL_DIR = DATA_ROOT / "validation"

MODEL_DIR = DATA_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

# MODEL_PATH = MODEL_DIR / "inat_resnet50_best.pth"

#processing

In [3]:
INPUT_SIZE = 320

# transforms.RandomResizedCrop(
#         350,
#         scale=(0.80, 1.0),       # at lease 80%
#         ratio=(0.85, 1.15),      # ratio limit
#         interpolation=transforms.InterpolationMode.BILINEAR,
#         antialias=True
#     ),

# transforms.Resize(INPUT_SIZE , interpolation=transforms.InterpolationMode.BILINEAR,
#        antialias=True),

train_transform = transforms.Compose([
    transforms.CenterCrop(INPUT_SIZE),
    
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# load traning and validation

In [4]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=val_transform
)

assert train_dataset.class_to_idx == val_dataset.class_to_idx, (
    "Train and validation folder differed"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,  
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=8,
    pin_memory=torch.cuda.is_available()
)

print("Classes:", len(train_dataset.classes))
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Class mapping:", train_dataset.class_to_idx)

Classes: 1000
Train images: 40000
Validation images: 10000
Class mapping: {'00001_Animalia_Annelida_Polychaeta_Sabellida_Sabellidae_Sabella_spallanzanii': 0, '00003_Animalia_Annelida_Polychaeta_Sabellida_Serpulidae_Spirobranchus_cariniferus': 1, '00004_Animalia_Arthropoda_Arachnida_Araneae_Agelenidae_Eratigena_duellica': 2, '00018_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Argiope_bruennichi': 3, '00024_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Cyclosa_turbinata': 4, '00030_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Larinioides_cornutus': 5, '00038_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Neoscona_crucifera': 6, '00052_Animalia_Arthropoda_Arachnida_Araneae_Corinnidae_Nyssus_coloripes': 7, '00055_Animalia_Arthropoda_Arachnida_Araneae_Filistatidae_Kukulcania_hibernalis': 8, '00061_Animalia_Arthropoda_Arachnida_Araneae_Oxyopidae_Oxyopes_salticus': 9, '00063_Animalia_Arthropoda_Arachnida_Araneae_Oxyopidae_Peucetia_viridans': 10, '00105_Animalia_Arthropoda_Arachnida_A

# training function

In [5]:
def training(
    EPOCHS,
    model,
    optimizer,
    scheduler,
    criterion,
    fname,
    classes
):
    best_val_accuracy = -1.0
    early_stop_patience = 7
    epochs_without_improvement = 0
    min_improvement = 1e-4

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    # save training
    history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rates": []
    }
    history_fname = str(Path(fname).with_suffix("")) + "_history.json"

    for epoch in range(EPOCHS):
        #  Training 
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * labels.size(0)
            train_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            train_total += labels.size(0)

        #  Validation 
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.inference_mode():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=use_amp
                ):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * labels.size(0)
                val_correct += (
                    outputs.argmax(dim=1) == labels
                ).sum().item()
                val_total += labels.size(0)

        train_loss /= train_total
        val_loss /= val_total
        train_accuracy = train_correct / train_total
        val_accuracy = val_correct / val_total

        scheduler.step(val_accuracy)

        learning_rates = [
            f"{group['lr']:.2e}"
            for group in optimizer.param_groups
        ]

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train acc: {train_accuracy:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val acc: {val_accuracy:.4f} | "
            f"LR: {learning_rates}"
        )
        
        # data adding
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        history["learning_rates"].append(
            [group["lr"] for group in optimizer.param_groups]
        )
        with open(history_fname, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=4)

        if val_accuracy > best_val_accuracy + min_improvement:
            best_val_accuracy = val_accuracy
            epochs_without_improvement = 0

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "classes": classes,
                "num_classes": len(classes),
                "best_val_accuracy": best_val_accuracy,
                "train_accuracy": train_accuracy,
                "val_accuracy": val_accuracy,
                "train_loss": train_loss,
                "val_loss": val_loss
            }, (fname+".pth"))

            print(
                f"Best model saved: {(fname+'.pth')} "
                f"(val_acc={best_val_accuracy:.4f})"
            )

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stop_patience:
            print(
                f"Early stopping. "
                f"Best val accuracy: {best_val_accuracy:.4f}"
            )
            break

    return best_val_accuracy

# load pretrained model

In [6]:
weights = ResNet18_Weights.IMAGENET1K_V1

pt_model = resnet18(weights=weights)

num_classes = len(train_dataset.classes)

in_features = pt_model.fc.in_features


pt_model.fc = nn.Linear(
    pt_model.fc.in_features,
    num_classes
)


##
early_parameters = [
    parameter
    for module in (
        pt_model.conv1,
        pt_model.bn1,
        pt_model.layer1,
        pt_model.layer2
    )
    for parameter in module.parameters()
]
## up
pt_model = pt_model.to(device)

pt_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

pt_optimizer = torch.optim.AdamW([
    {
        "params": early_parameters,
        "lr": 3e-6
    },
    {
        "params": pt_model.layer3.parameters(),
        "lr": 1e-5
    },
    {
        "params": pt_model.layer4.parameters(),
        "lr": 3e-5
    },
    {
        "params": pt_model.fc.parameters(),
        "lr": 1e-5
    }
], weight_decay=5e-4)


pt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    pt_optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    threshold=0.0005,
    threshold_mode="rel",
    cooldown=1,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
pt_model_filename = f"inat_resnet18_{num_classes}_ptclasses_imagenet"

training(50,pt_model,pt_optimizer,pt_scheduler,pt_criterion,pt_model_filename,train_dataset.classes)


Epoch 01/50 | Train loss: 5.9125 | Train acc: 0.0458 | Val loss: 5.4058 | Val acc: 0.1420 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet18_500_ptclasses_imagenet.pth (val_acc=0.1420)
Epoch 02/50 | Train loss: 5.0909 | Train acc: 0.2381 | Val loss: 4.7432 | Val acc: 0.2908 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet18_500_ptclasses_imagenet.pth (val_acc=0.2908)
Epoch 03/50 | Train loss: 4.5536 | Train acc: 0.3743 | Val loss: 4.3485 | Val acc: 0.3888 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet18_500_ptclasses_imagenet.pth (val_acc=0.3888)
Epoch 04/50 | Train loss: 4.1448 | Train acc: 0.4607 | Val loss: 3.9588 | Val acc: 0.4458 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet18_500_ptclasses_imagenet.pth (val_acc=0.4458)
Epoch 05/50 | Train loss: 3.8082 | Train acc: 0.5346 | Val loss: 3.7240 | Val acc: 0.5002 | LR: ['3.00e-06', '1.00e-

0.7138

In [7]:
weights = ResNet50_Weights.IMAGENET1K_V2

pt_model = resnet50(weights=weights)

num_classes = len(train_dataset.classes)

in_features = pt_model.fc.in_features


pt_model.fc = nn.Linear(
    pt_model.fc.in_features,
    num_classes
)


##
early_parameters = [
    parameter
    for module in (
        pt_model.conv1,
        pt_model.bn1,
        pt_model.layer1,
        pt_model.layer2
    )
    for parameter in module.parameters()
]
## up
pt_model = pt_model.to(device)

pt_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

pt_optimizer = torch.optim.AdamW([
    {
        "params": early_parameters,
        "lr": 3e-6
    },
    {
        "params": pt_model.layer3.parameters(),
        "lr": 1e-5
    },
    {
        "params": pt_model.layer4.parameters(),
        "lr": 3e-5
    },
    {
        "params": pt_model.fc.parameters(),
        "lr": 1e-5
    }
], weight_decay=5e-4)



pt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    pt_optimizer,
    mode="min",
    factor=0.5,             
    patience=3,             
    threshold=0.0005,
    threshold_mode="rel",
    cooldown=1,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
pt_model_filename = f"inat_resnet50_{num_classes}_ptclasses_imagenet"

training(50,pt_model,pt_optimizer,pt_scheduler,pt_criterion,pt_model_filename,train_dataset.classes)


Epoch 01/50 | Train loss: 5.9638 | Train acc: 0.0411 | Val loss: 5.2379 | Val acc: 0.1580 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth (val_acc=0.1580)
Epoch 02/50 | Train loss: 4.6765 | Train acc: 0.2905 | Val loss: 3.8234 | Val acc: 0.4194 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth (val_acc=0.4194)
Epoch 03/50 | Train loss: 3.5583 | Train acc: 0.5002 | Val loss: 3.0696 | Val acc: 0.5552 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth (val_acc=0.5552)
Epoch 04/50 | Train loss: 2.8904 | Train acc: 0.6244 | Val loss: 2.6461 | Val acc: 0.6330 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth (val_acc=0.6330)
Epoch 05/50 | Train loss: 2.4531 | Train acc: 0.7125 | Val loss: 2.3886 | Val acc: 0.6808 | LR: ['1.50e-06', '5.00e-

0.785

# train with no preset weights

In [8]:
np_model = resnet18(weights=None)

num_classes = len(train_dataset.classes)

np_model.fc = nn.Linear(
    np_model.fc.in_features,
    num_classes
)

np_model = np_model.to(device)

np_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

np_optimizer = torch.optim.AdamW(
    np_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)



np_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    np_optimizer,
    mode="max",       
    factor=0.5,      
    patience=2,       
    threshold=1e-3,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
np_model_filename = f"inat_resnet18_{num_classes}_npclasses_imagenet"

training(50,np_model,np_optimizer,np_scheduler,np_criterion,np_model_filename,train_dataset.classes)

Epoch 01/50 | Train loss: 5.9850 | Train acc: 0.0144 | Val loss: 5.7385 | Val acc: 0.0248 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0248)
Epoch 02/50 | Train loss: 5.6015 | Train acc: 0.0308 | Val loss: 5.5338 | Val acc: 0.0380 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0380)
Epoch 03/50 | Train loss: 5.3713 | Train acc: 0.0479 | Val loss: 5.4375 | Val acc: 0.0458 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0458)
Epoch 04/50 | Train loss: 5.1598 | Train acc: 0.0722 | Val loss: 5.1699 | Val acc: 0.0750 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0750)
Epoch 05/50 | Train loss: 4.9736 | Train acc: 0.0942 | Val loss: 5.0513 | Val acc: 0.0962 | LR: ['1.00e-04']
Best model saved: inat_resnet18_500_npclasses_imagenet.pth (val_acc=0.0962)
Epoch 06/50 | Train loss: 4.7952 | Train acc: 0.1177 | Val loss: 4.8675 | V

0.3558

In [10]:

np_model = resnet50(weights=None)

num_classes = len(train_dataset.classes)

np_model.fc = nn.Linear(
    np_model.fc.in_features,
    num_classes
)

np_model = np_model.to(device)

np_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

np_optimizer = torch.optim.AdamW(
    np_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)



np_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    np_optimizer,
    mode="max",       
    factor=0.5,       
    patience=2,       
    threshold=1e-3,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
np_model_filename = f"inat_resnet50_{num_classes}_npclasses_imagenet"

training(50,np_model,np_optimizer,np_scheduler,np_criterion,np_model_filename,train_dataset.classes)

Epoch 01/50 | Train loss: 6.1583 | Train acc: 0.0054 | Val loss: 6.1023 | Val acc: 0.0108 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0108)
Epoch 02/50 | Train loss: 5.8036 | Train acc: 0.0156 | Val loss: 5.7200 | Val acc: 0.0220 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0220)
Epoch 03/50 | Train loss: 5.6118 | Train acc: 0.0257 | Val loss: 5.5506 | Val acc: 0.0326 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0326)
Epoch 04/50 | Train loss: 5.4625 | Train acc: 0.0360 | Val loss: 5.4454 | Val acc: 0.0428 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0428)
Epoch 05/50 | Train loss: 5.3101 | Train acc: 0.0490 | Val loss: 5.3164 | Val acc: 0.0648 | LR: ['1.00e-04']
Best model saved: inat_resnet50_500_npclasses_imagenet.pth (val_acc=0.0648)
Epoch 06/50 | Train loss: 5.1740 | Train acc: 0.0653 | Val loss: 5.1759 | V

0.3338

# model reloading

In [10]:
# # ==================== Resume training ====================

# pt_model_filename= Path("inat_resnet50_500_ptclasses_imagenet.pth")
# ADDITIONAL_EPOCHS = 20   
# LR_FACTOR = 0.3          

# checkpoint = torch.load(
#     pt_model_filename,
#     map_location=device,
#     weights_only=True
# )


# assert checkpoint["classes"] == train_dataset.classes
# assert checkpoint["num_classes"] == len(train_dataset.classes)

# # recover
# pt_model.load_state_dict(
#     checkpoint["model_state_dict"]
# )

# pt_optimizer.load_state_dict(
#     checkpoint["optimizer_state_dict"]
# )

# pt_scheduler.load_state_dict(
#     checkpoint["scheduler_state_dict"]
# )

# # make sure optizmizer is right device
# for state in pt_optimizer.state.values():
#     for key, value in state.items():
#         if torch.is_tensor(value):
#             state[key] = value.to(device)

# start_epoch = checkpoint["epoch"]
# best_val_accuracy = checkpoint["best_val_accuracy"]


# for group in pt_optimizer.param_groups:
#     group["lr"] *= LR_FACTOR

# total_epochs = start_epoch + ADDITIONAL_EPOCHS

# print(f"Loaded: {pt_model_filename}")
# print(f"Resume from epoch: {start_epoch + 1}")
# print(f"Train until epoch: {total_epochs}")
# print(f"Previous best val accuracy: {best_val_accuracy:.4f}")
# print("Learning rates:", [
#     group["lr"]
#     for group in pt_optimizer.param_groups
# ])

# best_accuracy = training(
#     total_epochs,
#     pt_model,
#     pt_optimizer,
#     pt_scheduler,
#     pt_criterion,
#     pt_model_filename,
#     train_dataset.classes,
# )

Loaded: inat_resnet50_500_ptclasses_imagenet.pth
Resume from epoch: 32
Train until epoch: 51
Previous best val accuracy: 0.7850
Learning rates: [3e-08, 4.6875e-08, 1.40625e-07, 4.6875e-08]


Epoch 01/51 | Train loss: 1.3928 | Train acc: 0.9444 | Val loss: 1.8751 | Val acc: 0.7820 | LR: ['3.00e-08', '4.69e-08', '1.41e-07', '4.69e-08']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth.pth (val_acc=0.7820)
Epoch 02/51 | Train loss: 1.3919 | Train acc: 0.9439 | Val loss: 1.8739 | Val acc: 0.7828 | LR: ['3.00e-08', '4.69e-08', '1.41e-07', '4.69e-08']
Best model saved: inat_resnet50_500_ptclasses_imagenet.pth.pth (val_acc=0.7828)
Epoch 03/51 | Train loss: 1.3910 | Train acc: 0.9445 | Val loss: 1.8823 | Val acc: 0.7810 | LR: ['3.00e-08', '4.69e-08', '1.41e-07', '4.69e-08']
Epoch 04/51 | Train loss: 1.3887 | Train acc: 0.9466 | Val loss: 1.8773 | Val acc: 0.7820 | LR: ['3.00e-08', '4.69e-08', '1.00e-07', '4.69e-08']
Epoch 05/51 | Train loss: 1.3899 | Train acc: 0.9445 | Val loss: 1.8743 | Val acc: 0.7794 | LR: ['3.00e-08', '4.69e-08', '1.00e-07', '4.69e-08']
Epoch 06/51 | Train loss: 1.3917 | Train acc: 0.9442 | Val loss: 1.8846 | Val acc: 0.7788 | LR: ['3.00e-08', '4.69e

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights,resnet101, ResNet101_Weights
weights = ResNet101_Weights.IMAGENET1K_V2

pt_model = resnet101(weights=weights)

# Get the associated image preprocessing transforms
preprocess = weights.transforms()

num_classes = len(train_dataset.classes)

in_features = pt_model.fc.in_features


pt_model.fc = nn.Linear(
    pt_model.fc.in_features,
    num_classes
)


##
early_parameters = [
    parameter
    for module in (
        pt_model.conv1,
        pt_model.bn1,
        pt_model.layer1,
        pt_model.layer2
    )
    for parameter in module.parameters()
]
## up
pt_model = pt_model.to(device)

pt_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

pt_optimizer = torch.optim.AdamW([
    {
        "params": early_parameters,
        "lr": 3e-6
    },
    {
        "params": pt_model.layer3.parameters(),
        "lr": 1e-5
    },
    {
        "params": pt_model.layer4.parameters(),
        "lr": 3e-5
    },
    {
        "params": pt_model.fc.parameters(),
        "lr": 1e-5
    }
], weight_decay=5e-4)



pt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    pt_optimizer,
    mode="min",
    factor=0.5,             
    patience=3,             
    threshold=0.0005,
    threshold_mode="rel",
    cooldown=1,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
pt_model_filename = f"inat_resnet101_{num_classes}_ptclasses_imagenet"

training(50,pt_model,pt_optimizer,pt_scheduler,pt_criterion,pt_model_filename,train_dataset.classes)


Epoch 01/50 | Train loss: 6.3760 | Train acc: 0.0517 | Val loss: 5.0280 | Val acc: 0.2158 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet101_1000_ptclasses_imagenet.pth (val_acc=0.2158)
Epoch 02/50 | Train loss: 4.2387 | Train acc: 0.3512 | Val loss: 3.3144 | Val acc: 0.4888 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet101_1000_ptclasses_imagenet.pth (val_acc=0.4888)
Epoch 03/50 | Train loss: 3.0611 | Train acc: 0.5676 | Val loss: 2.7456 | Val acc: 0.6032 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet101_1000_ptclasses_imagenet.pth (val_acc=0.6032)
Epoch 04/50 | Train loss: 2.5167 | Train acc: 0.6852 | Val loss: 2.4673 | Val acc: 0.6622 | LR: ['3.00e-06', '1.00e-05', '3.00e-05', '1.00e-05']
Best model saved: inat_resnet101_1000_ptclasses_imagenet.pth (val_acc=0.6622)
Epoch 05/50 | Train loss: 2.1941 | Train acc: 0.7552 | Val loss: 2.2966 | Val acc: 0.6990 | LR: ['1.50e-06',